# 07 · Multi-agent basics

**Course:** Building LLM Agents, notebook 7 of 10
**Where we are:** for six notebooks we have built and hardened one agent. Now we ask a new question. What happens when you have several agents, each with its own role, and let them work together?

### What you'll learn
- Why you would ever want more than one agent, and the honest cost of doing so
- The key idea: an agent is just an LLM with a role, so "multi-agent" is simply routing one agent's output into another's input, in plain Python with no framework
- Pattern 1, sequential handoff: a `Researcher` gathers notes, a `Writer` turns them into prose
- Pattern 2, generator/critic: a `Writer` drafts, a `Critic` reviews, the `Writer` revises, and you see the quality lift
- A tiny helper to print "AgentName: ..." turns so a multi-agent exchange stays readable

### Prerequisites
- Notebooks 04 and 06: you know what an agent is, an LLM plus a role (system prompt) plus tools plus a loop
- You can `from helpers import client, MODEL, Agent, Tool`

> Mental model: you already have the only building block you need, a single agent. A multi-agent system is not a new kind of object. It is the same `Agent` class, instantiated a few times with different roles, where the text one agent produces becomes the prompt the next one reads. That is it. The cleverness is in the wiring, and the wiring is ordinary Python.

> Recap of notebook 06: you built a robust single agent, with a multi-tool toolbox, error handling that feeds failures back to the model, guardrails, and a reliable system prompt. We now keep that exact `Agent` (from `helpers.py`) and ask when one well-built agent is not the right shape, and what we do instead.

## 0. Setup

Same as every notebook: import the ready-made client and the reusable `Agent` and `Tool` you built in notebook 04. Nothing new is hidden in here. The `Agent` is the plain reason, act, observe loop from earlier.

In [ ]:
from helpers import client, MODEL, Agent, Tool, show_messages
print("Model:", MODEL)

## 1. Why more than one agent?

You could stuff every responsibility into a single agent: research the topic, then write a polished blog post, then proofread it, then check it for legal issues. It often works. But as a task grows, one agent starts to struggle for the same reasons a single person juggling four jobs does. Splitting into focused roles buys you:

- Separation of concerns. Each agent does one thing. A `Researcher` researches; a `Writer` writes. That is easier to reason about and easier to swap out.
- Shorter, focused context. Each agent only sees what its job needs, instead of one giant prompt mixing research instructions, style rules, and legal policy. Shorter context is cheaper, faster, and less likely to make the model lose the thread.
- Specialised personas. You can give the `Critic` a genuinely demanding, sceptical system prompt without making your `Writer` timid. Different roles want different attitudes.
- Different tools per role. The `Researcher` gets a search tool; the `Writer` gets none, because it should not be inventing facts; a `Coder` gets a code-runner. Least privilege, per role.
- Testable in isolation. You can run the `Critic` on a fixed draft and check its feedback, without re-running the whole pipeline.

> Cost, be honest: more agents is not automatically better. Each agent is its own LLM call (often several, if it loops), each resending its own context. Two agents can easily cost more than twice a single one once you add the handoff messages. And every handoff is a place where information can get garbled. Add an agent only when the role is genuinely distinct: a different job, a different persona, or a different toolset. If you cannot say in one sentence why a second agent earns its keep, use one.

## 2. The key idea: outputs become inputs

Here is the whole trick, stripped bare. An agent's `.run(task)` takes a string in and returns a string out:

```
notes  = researcher.run("Research X")     # text out
post   = writer.run("Write using: " + notes)   # ...becomes text in
```

"Agents talking to each other" sounds mystical. It isn't. You take the first agent's returned text and paste it into the second agent's prompt. The agents never magically share a brain. You are the message bus. There is no framework and no special protocol. We can make that literal with a tiny print helper so a conversation between agents reads like a conversation.

In [ ]:
def say(name, text):
    """Print one agent's turn, the way a chat log would show it."""
    print(f"\n{'='*70}\n{name}:\n{'-'*70}\n{text.strip()}")

> Key idea: a multi-agent system is a small program where the values flowing between functions happen to be natural-language text, and the functions happen to be LLM calls. Once you see it that way, you can wire agents together with the same `if`, `for`, and variable-passing you already know.

![Sequential handoff: one agent's text output becomes another's input](figures/07-handoff.svg)

*Figure: Researcher (notes) Writer. One agent's output becomes the next one's input. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 3. Pattern 1: sequential handoff (two specialists)

The simplest multi-agent shape: A does its job, hands the result to B, and B finishes. We will build a two-stage content pipeline:

1. a `Researcher` that gathers facts and returns tidy notes (we will give it a mock search tool so it has something to "look up"), then
2. a `Writer` that takes those notes and produces the final prose. The writer has no search tool on purpose: its only job is to write from the notes it is given, not to invent.

First, a mock search tool. In production this would hit a real search API or your documents (notebook 05 showed lexical retrieval); here we hard-code a tiny knowledge base so the lesson is about the handoff, not the search.

In [ ]:
# A pretend search index. Real systems would query the web or your own docs.
_FACTS = {
    "puffin": [
        "Atlantic puffins spend most of the year at sea, coming ashore only to breed.",
        "They can carry a dozen or more small fish crosswise in their beak at once.",
        "Norway's Rost archipelago hosts one of the world's largest puffin colonies.",
    ],
    "fjord": [
        "A fjord is a long, deep inlet carved by glaciers and flooded by the sea.",
        "Norway's Sognefjord is over 200 km long and more than 1,300 m deep.",
    ],
}

def mock_search(query: str) -> str:
    """Look up facts about a topic. Returns bullet-point notes."""
    q = query.lower()
    for key, facts in _FACTS.items():
        if key in q:
            return "\n".join(f"- {f}" for f in facts)
    return "- (no facts found for that query)"

search_tool = Tool(
    mock_search,
    description="Search a knowledge base for facts about a topic. Pass a short topic keyword.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Topic keyword, e.g. 'puffin'."}},
        "required": ["query"],
    },
)

Now the two agents. Notice how different their system prompts are. That difference is the whole point. The researcher is told to only report what it finds; the writer is told to write only from the notes it is handed.

In [ ]:
researcher = Agent(
    name="Researcher",
    system=(
        "You are a research assistant. Use the search tool to gather facts about the "
        "topic, then return a short, neutral bullet-point list of the key facts you found. "
        "Do NOT write prose and do NOT add facts you did not find via the tool."
    ),
    tools=[search_tool],
    temperature=0,   # gathering facts is a decision task: keep it predictable
)

writer = Agent(
    name="Writer",
    system=(
        "You are a science writer for a general audience. You will be given research "
        "notes. Write a single vivid paragraph (3-4 sentences) based ONLY on those notes. "
        "Do not invent facts beyond the notes."
    ),
    tools=[],         # the writer has no tools: it must write from what it's given
    temperature=0.7,  # writing is creative, so we allow some warmth here
)

Now the handoff itself, the part worth slowing down on. It is three lines: run the researcher, pass its output into the writer's prompt, run the writer.

In [ ]:
topic = "puffin"

# Stage 1: the Researcher gathers notes (it will call the search tool internally).
notes = researcher.run(f"Research this topic and list the key facts: {topic}")
say("Researcher", notes)

# Stage 2: THE HANDOFF. The researcher's text becomes part of the writer's prompt.
writer_prompt = f"Here are the research notes:\n\n{notes}\n\nWrite the paragraph."
article = writer.run(writer_prompt)
say("Writer", article)

> What just happened: two separate LLMs, two separate roles, two separate contexts. The only thing connecting them is the variable `notes`, produced by one agent and spliced into the prompt of the other. There is no shared memory and no magic. You routed the message. That is a multi-agent system.

> Rule of thumb: give each agent the narrowest role and the fewest tools it needs. The writer having no search tool is not a limitation, it is a guardrail. It physically cannot wander off and invent "facts," because it can only work with the notes you handed it.

## 4. Pattern 2: the generator/critic pair

This is the single most useful starter pattern, and the one most worth your time. The idea is the way good writing actually gets made: someone drafts, someone else with high standards and fresh eyes critiques, then the draft is revised. Two roles:

- a `Writer` (generator) that produces a draft, and
- a `Critic` that does not rewrite, but lists concrete, specific improvements.

Then the writer revises using that feedback. We will do this explicitly, one round at a time, and print every stage so you can see the draft get better. The lift comes from the critic having a different system prompt, demanding and specific and allergic to vagueness, which a single agent grading its own work tends not to apply to itself.

Let's reuse our `writer` from above and add a sharp `critic`.

![Generator/critic loop: draft to critique to revise, capped](figures/07-generator-critic.svg)

*Figure: draft, critique, revise, over a bounded number of rounds. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
critic = Agent(
    name="Critic",
    system=(
        "You are a tough but fair editor. You are given a short piece of writing. "
        "Do NOT rewrite it. Instead, list 2-3 specific, actionable improvements as bullet "
        "points (e.g. 'the opening is generic - lead with the most surprising fact'). "
        "Be concrete. If something is genuinely good, you may say so briefly, but always "
        "give at least two improvements."
    ),
    tools=[],
    temperature=0,   # judging is a decision task -> keep it focused and consistent
)

**Round 0, the first draft.** A fresh, deliberately quick draft so we have room to improve. (We re-run the writer on a simple brief rather than reuse the handoff paragraph, to keep this section self-contained.)

In [ ]:
brief = "Write a short, punchy paragraph (3-4 sentences) introducing the Atlantic puffin to curious readers."

draft = writer.run(brief)
say("Writer (draft)", draft)

**The critique.** We hand the draft straight into the critic's prompt, the same output-becomes-input move as before. The critic returns feedback, not a rewrite.

In [ ]:
critique = critic.run(f"Here is the draft to review:\n\n{draft}")
say("Critic", critique)

**Round 1, the revision.** Now we give the writer both its own draft and the critique, and ask for an improved version. This is the loop closing: the critic's words steer the writer's next attempt.

In [ ]:
revision_prompt = (
    f"Here is your previous draft:\n\n{draft}\n\n"
    f"An editor gave this feedback:\n\n{critique}\n\n"
    "Rewrite the paragraph, addressing every point of the feedback. "
    "Keep it to 3-4 sentences."
)
revised = writer.run(revision_prompt)
say("Writer (revised)", revised)

Read the draft and the revision side by side: the revised version should visibly fold in the critic's points. That improvement did not come from a smarter model. It came from collaboration between two roles. A single agent rarely critiques itself this honestly; a dedicated critic with high standards does.

> Key idea: the generator/critic split works because the two roles have different incentives baked into their prompts. The writer wants to produce; the critic wants to find fault. Holding those in tension, across two agents, is what lifts quality.

### Doing it twice: wrapping the round in a function

One round usually helps the most; a second can polish further, with diminishing returns and rising cost. Because it is just message-passing, we can wrap "critique, then revise" in a plain Python function and call it in a loop. Notice there is still no framework here, only a `for` loop over ordinary function calls.

In [ ]:
def critique_and_revise(text, rounds=2):
    """Run the generator/critic loop for a fixed number of rounds."""
    current = text
    for i in range(1, rounds + 1):
        fb = critic.run(f"Here is the draft to review:\n\n{current}")
        say(f"Critic (round {i})", fb)
        current = writer.run(
            f"Here is your previous draft:\n\n{current}\n\n"
            f"An editor gave this feedback:\n\n{fb}\n\n"
            "Rewrite it, addressing every point. Keep it to 3-4 sentences."
        )
        say(f"Writer (round {i})", current)
    return current

final = critique_and_revise(draft, rounds=2)
say("FINAL", final)

> Gotcha, know when to stop: the loop above runs a fixed number of rounds. Left unbounded, a writer and critic will happily critique forever (the critic's prompt always demands improvements), burning tokens for ever-smaller gains. In real systems you cap the rounds, or let the critic emit a clear "no further changes needed" signal that breaks the loop. This is the same `max_steps` discipline from the agent loop in notebook 04, just one level up, at the level of agents rather than steps.

## 5. The "agents talking" mechanic, made explicit

Strip away the topic and the personas, and every multi-agent system in this notebook is the same two moves:

1. Run an agent and get a string back.
2. Splice that string into the next agent's prompt, then run that one.

That is the message bus, and it is you. The diagram below is the whole notebook:

```
            handoff (pattern 1)                 critique loop (pattern 2)

  Researcher ──notes──▶ Writer            Writer ──draft──▶ Critic
                                            ▲                  │
                                            └──feedback────────┘
                                              (you route it back)
```

Everything fancier, deciding which agent runs next, running several in parallel, having a "manager" agent delegate, is just more elaborate routing of the same strings. Choosing that routing is a design decision with a name: orchestration. That is notebook 08.

#TODO add figure instead 

> Why you care: once you internalise that agents are functions and messages are their arguments, you stop reaching for heavy frameworks to do simple things. A two-agent pipeline is ten lines of Python. Frameworks earn their place when the routing gets complicated, with many agents, conditional handoffs, retries, and parallelism, and that is exactly the gap notebooks 08 and 09 fill.

## Recap

You can now:
- explain why to split work across agents (focus, shorter context, distinct personas, per-role tools, isolated testing) and when not to (the token and coordination cost)
- see that "multi-agent" is just one agent's output becoming another's input, in plain Python with no framework
- build a sequential handoff: `Researcher` (with a search tool) to `Writer` (no tools, writes from the notes)
- build a generator/critic pair and run the draft, critique, revise loop, watching quality rise
- wrap the loop in a function with a hard round cap so it cannot run away

### Try it yourself
1. New pipeline. Add a third agent, a `FactChecker`, after the writer in pattern 1. Give it the notes and the article and have it flag any claim in the article not supported by the notes.
2. Tune the critic. Make the critic's persona harsher ("ruthless editor of a prestige magazine") and softer ("encouraging writing coach"). How does the revised paragraph change?
3. Stopping signal. Modify `critique_and_revise` so the critic can output the exact token `DONE` when the draft is good enough, and break the loop when it does.
4. Swap the topic. Run the handoff on `"fjord"` instead of `"puffin"`. Then try a topic the mock search does not know. What does the writer do, and is that a problem? (Hint: it is why the writer has no search tool.)

### Next up
**08 · Orchestration patterns.** So far you decided, by hand, who runs when. With more than two agents that quickly gets unwieldy. Next we name the standard ways to wire agents together (sequential pipeline, router/dispatcher, orchestrator-worker, and parallel fan-out) and let code decide the routing for us.